# Exercise 6 

Goal: given one specific car, return a ranked list of similar cars.


In [1]:
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler

DATA_PATH = Path('data.csv')
df_raw = pd.read_csv(DATA_PATH)
df_raw = df_raw.reset_index().rename(columns={'index': 'row_id'})

print(df_raw.shape)
display(df_raw.head())

(11914, 17)


,row_id,Make,Model,Year,Engine Fuel Type,Engine HP,Engine Cylinders,Transmission Type,Driven_Wheels,Number of Doors,Market Category,Vehicle Size,Vehicle Style,highway MPG,city mpg,Popularity,MSRP
0,0,BMW,1 Series M,2011,premium unleaded (required),335.0,6.0,MANUAL,rear wheel drive,2.0,"Factory Tuner,Luxury,High-Performance",Compact,Coupe,26,19,3916,46135
1,1,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Convertible,28,19,3916,40650
2,2,BMW,1 Series,2011,premium unleaded (required),300.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,High-Performance",Compact,Coupe,28,20,3916,36350
3,3,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,"Luxury,Performance",Compact,Coupe,28,18,3916,29450
4,4,BMW,1 Series,2011,premium unleaded (required),230.0,6.0,MANUAL,rear wheel drive,2.0,Luxury,Compact,Convertible,28,18,3916,34500


The dataset contains car specifications such as horsepower, MPG, MSRP, body style, size, drivetrain, and fuel type. Missing numeric values are filled with medians. Missing categorical values are filled with `Unknown`.

In [2]:
df = df_raw.copy()

numeric_cols = [
    'Year', 'Engine HP', 'Engine Cylinders', 'Number of Doors',
    'highway MPG', 'city mpg', 'Popularity', 'MSRP'
]

categorical_cols = [
    'Make', 'Engine Fuel Type', 'Transmission Type', 'Driven_Wheels',
    'Vehicle Size', 'Vehicle Style'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    df[col] = df[col].fillna('Unknown').astype(str)

df['Market Category'] = df['Market Category'].fillna('Unknown').astype(str)

df['car_label'] = df.apply(
    lambda row: f"{row['row_id']}: {row['Make']} {row['Model']} ({row['Year']})",
    axis=1
)

print('Missing values after cleaning:', int(df.isna().sum().sum()))
display(df[['row_id', 'car_label', 'Vehicle Style', 'Vehicle Size', 'Engine HP', 'highway MPG', 'city mpg', 'MSRP']].head(10))

Missing values after cleaning: 0


,row_id,car_label,Vehicle Style,Vehicle Size,Engine HP,highway MPG,city mpg,MSRP
0,0,0: BMW 1 Series M (2011),Coupe,Compact,335.0,26,19,46135
1,1,1: BMW 1 Series (2011),Convertible,Compact,300.0,28,19,40650
2,2,2: BMW 1 Series (2011),Coupe,Compact,300.0,28,20,36350
3,3,3: BMW 1 Series (2011),Coupe,Compact,230.0,28,18,29450
4,4,4: BMW 1 Series (2011),Convertible,Compact,230.0,28,18,34500
5,5,5: BMW 1 Series (2012),Coupe,Compact,230.0,28,18,31200
6,6,6: BMW 1 Series (2012),Convertible,Compact,300.0,26,17,44100
7,7,7: BMW 1 Series (2012),Coupe,Compact,300.0,28,20,39300
8,8,8: BMW 1 Series (2012),Convertible,Compact,230.0,28,18,36900
9,9,9: BMW 1 Series (2013),Convertible,Compact,230.0,27,18,37200


A content-based recommender needs features that describe the car itself. I use:

- numeric specs: year, horsepower, cylinders, doors, MPG, popularity, MSRP,
- categorical specs: fuel type, transmission, drivetrain, size, body style,
- multi-label market categories: luxury, performance, crossover, etc.,
- make/brand with a low weight so the system can recommend similar competitors, not only same-brand cars.

In [3]:
def build_feature_matrix(df):
    numeric = df[numeric_cols].copy()
    numeric_scaled = pd.DataFrame(
        StandardScaler().fit_transform(numeric),
        columns=numeric_cols,
        index=df.index,
    )

    # Price and horsepower can be very skewed. A log version helps compare realistic ranges.
    numeric_scaled['log_MSRP'] = StandardScaler().fit_transform(np.log1p(df[['MSRP']]))[:, 0]
    numeric_scaled['log_Engine_HP'] = StandardScaler().fit_transform(np.log1p(df[['Engine HP']]))[:, 0]

    make_features = pd.get_dummies(df['Make'], prefix='make', dtype=float)
    spec_categories = pd.get_dummies(
        df[['Engine Fuel Type', 'Transmission Type', 'Driven_Wheels', 'Vehicle Size', 'Vehicle Style']],
        dtype=float,
    )

    market_tokens = (
        df['Market Category']
        .str.get_dummies(sep=',')
        .rename(columns=lambda name: f"market_{name.strip().replace(' ', '_')}")
        .astype(float)
    )

    feature_parts = {
        'numeric': numeric_scaled,
        'make': make_features,
        'spec_categories': spec_categories,
        'market': market_tokens,
    }

    # Weights encode what should matter most for a human-sensible car recommendation.
    weights = {
        'numeric': 1.25,
        'make': 0.25,
        'spec_categories': 1.40,
        'market': 1.10,
    }

    weighted_parts = []
    for group_name, part in feature_parts.items():
        weighted_parts.append(part * weights[group_name])

    features = pd.concat(weighted_parts, axis=1).fillna(0.0)
    return features, weights

features, feature_weights = build_feature_matrix(df)
similarity_matrix = cosine_similarity(features)

print('Feature matrix shape:', features.shape)
print('Feature weights:', feature_weights)

Feature matrix shape: (11914, 108)
Feature weights: {'numeric': 1.25, 'make': 0.25, 'spec_categories': 1.4, 'market': 1.1}


`row_id` is the safest input because the same model name appears many times with different trims, years, engines, and prices.

In [4]:
display_cols = [
    'row_id', 'Make', 'Model', 'Year', 'Vehicle Style', 'Vehicle Size',
    'Engine Fuel Type', 'Transmission Type', 'Driven_Wheels',
    'Engine HP', 'Engine Cylinders', 'highway MPG', 'city mpg', 'MSRP'
]

def find_cars(model=None, make=None, year=None, limit=15):
    matches = pd.Series(True, index=df.index)
    exact_model = pd.Series(False, index=df.index)
    if model is not None:
        model_text = df['Model'].fillna('').astype(str)
        exact_model = model_text.str.lower().eq(str(model).lower())
        matches &= exact_model | model_text.str.contains(model, case=False, na=False)
    if make is not None:
        matches &= df['Make'].str.contains(make, case=False, na=False)
    if year is not None:
        matches &= df['Year'].eq(year)

    result = df.loc[matches, ['car_label'] + display_cols].copy()
    if model is not None and not result.empty:
        result['_exact_model'] = result['Model'].str.lower().eq(str(model).lower())
        result = result.sort_values(['_exact_model', 'Year', 'MSRP'], ascending=[False, False, True])
        result = result.drop(columns=['_exact_model'])
    return result.head(limit)

def recommend_by_row_id(row_id, top_n=10, unique_models=True):
    if row_id not in set(df['row_id']):
        raise ValueError(f'Unknown row_id: {row_id}')

    target_idx = int(df.index[df['row_id'].eq(row_id)][0])
    target = df.loc[target_idx]
    scores = list(enumerate(similarity_matrix[target_idx]))
    scores = sorted(scores, key=lambda item: item[1], reverse=True)

    selected = []
    seen_model_keys = set()
    for candidate_idx, score in scores:
        if candidate_idx == target_idx:
            continue

        candidate = df.loc[candidate_idx]
        model_key = (candidate['Make'], candidate['Model'])
        target_key = (target['Make'], target['Model'])

        if unique_models and model_key == target_key:
            continue
        if unique_models and model_key in seen_model_keys:
            continue

        seen_model_keys.add(model_key)
        selected.append((candidate_idx, score))
        if len(selected) == top_n:
            break

    result = df.loc[[idx for idx, _ in selected], display_cols].copy()
    result.insert(0, 'similarity', [score for _, score in selected])
    result.insert(0, 'input_car', target['car_label'])
    return result.reset_index(drop=True)

def recommend_by_search(model, make=None, year=None, top_n=10):
    matches = find_cars(model=model, make=make, year=year, limit=20)
    if matches.empty:
        raise ValueError('No matching car found. Try a broader search.')
    row_id = int(matches.iloc[0]['row_id'])
    print('Using input car:', matches.iloc[0]['car_label'])
    return recommend_by_row_id(row_id, top_n=top_n)

display(find_cars(model='Corolla', make='Toyota', limit=5))

,car_label,row_id,Make,Model,Year,Vehicle Style,Vehicle Size,Engine Fuel Type,Transmission Type,Driven_Wheels,Engine HP,Engine Cylinders,highway MPG,city mpg,MSRP
2951,2951: Toyota Corolla (2017),2951,Toyota,Corolla,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,132.0,4.0,36,28,18500
2956,2956: Toyota Corolla (2017),2956,Toyota,Corolla,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,132.0,4.0,36,28,18935
2953,2953: Toyota Corolla (2017),2953,Toyota,Corolla,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,140.0,4.0,40,30,19335
2957,2957: Toyota Corolla (2017),2957,Toyota,Corolla,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,132.0,4.0,35,28,20445
2950,2950: Toyota Corolla (2017),2950,Toyota,Corolla,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,140.0,4.0,40,30,21300


## 4. Example Recommendations

In [5]:
examples = [
    ('Odyssey', 'Honda'),
    ('Corolla', 'Toyota'),
    ('Mustang', 'Ford'),
    ('F-150', 'Ford'),
    ('911', 'Porsche'),
]

for model, make in examples:
    print('\n' + '=' * 90)
    print(f'Recommendations for {make} {model}')
    display(recommend_by_search(model=model, make=make, top_n=7))


Recommendations for Honda Odyssey
Using input car: 7305: Honda Odyssey (2016)


,input_car,similarity,row_id,Make,Model,Year,Vehicle Style,Vehicle Size,Engine Fuel Type,Transmission Type,Driven_Wheels,Engine HP,Engine Cylinders,highway MPG,city mpg,MSRP
0,7305: Honda Odyssey (2016),0.983608,9227,Toyota,Sienna,2016,Passenger Minivan,Large,regular unleaded,AUTOMATIC,front wheel drive,266.0,6.0,25,18,28850
1,7305: Honda Odyssey (2016),0.841561,7911,Nissan,Quest,2016,Passenger Minivan,Compact,regular unleaded,AUTOMATIC,front wheel drive,260.0,6.0,27,20,30540
2,7305: Honda Odyssey (2016),0.841518,8505,Honda,Ridgeline,2017,Crew Cab Pickup,Large,regular unleaded,AUTOMATIC,front wheel drive,280.0,6.0,26,19,29475
3,7305: Honda Odyssey (2016),0.825014,5499,Dodge,Grand Caravan,2017,Passenger Minivan,Midsize,regular unleaded,AUTOMATIC,front wheel drive,283.0,6.0,25,17,29395
4,7305: Honda Odyssey (2016),0.823785,9085,Kia,Sedona,2017,Passenger Minivan,Midsize,regular unleaded,AUTOMATIC,front wheel drive,276.0,6.0,25,18,36900
5,7305: Honda Odyssey (2016),0.816548,2335,Dodge,Caravan,2006,Passenger Minivan,Large,regular unleaded,AUTOMATIC,front wheel drive,180.0,6.0,24,17,22520
6,7305: Honda Odyssey (2016),0.796361,7411,Chrysler,Pacifica,2017,Passenger Minivan,Midsize,regular unleaded,AUTOMATIC,front wheel drive,287.0,6.0,28,18,30495



Recommendations for Toyota Corolla
Using input car: 2951: Toyota Corolla (2017)


,input_car,similarity,row_id,Make,Model,Year,Vehicle Style,Vehicle Size,Engine Fuel Type,Transmission Type,Driven_Wheels,Engine HP,Engine Cylinders,highway MPG,city mpg,MSRP
0,2951: Toyota Corolla (2017),0.994714,8525,Kia,Rio,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,138.0,4.0,36,27,17755
1,2951: Toyota Corolla (2017),0.990256,4879,Kia,Forte,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,147.0,4.0,38,29,19200
2,2951: Toyota Corolla (2017),0.988963,9789,Chevrolet,Sonic,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,138.0,4.0,36,27,20415
3,2951: Toyota Corolla (2017),0.988324,11357,Nissan,Versa,2017,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,109.0,4.0,39,31,17280
4,2951: Toyota Corolla (2017),0.987288,1183,Hyundai,Accent,2016,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,137.0,4.0,37,26,15745
5,2951: Toyota Corolla (2017),0.986988,2559,Honda,Civic,2015,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,143.0,4.0,39,30,19290
6,2951: Toyota Corolla (2017),0.986988,3866,Hyundai,Elantra,2016,Sedan,Compact,regular unleaded,AUTOMATIC,front wheel drive,145.0,4.0,37,27,21700



Recommendations for Ford Mustang
Using input car: 7088: Ford Mustang (2017)


,input_car,similarity,row_id,Make,Model,Year,Vehicle Style,Vehicle Size,Engine Fuel Type,Transmission Type,Driven_Wheels,Engine HP,Engine Cylinders,highway MPG,city mpg,MSRP
0,7088: Ford Mustang (2017),0.798704,8364,Ford,Ranger,2011,Extended Cab Pickup,Compact,regular unleaded,MANUAL,rear wheel drive,207.0,6.0,21,16,22425
1,7088: Ford Mustang (2017),0.788468,4390,Ford,Explorer Sport,2003,2dr SUV,Midsize,regular unleaded,MANUAL,rear wheel drive,203.0,6.0,20,15,21870
2,7088: Ford Mustang (2017),0.767205,9171,Ford,Shelby GT350,2016,Coupe,Midsize,premium unleaded (recommended),MANUAL,rear wheel drive,526.0,8.0,21,14,47795
3,7088: Ford Mustang (2017),0.754204,10724,Ford,Transit Wagon,2017,Passenger Van,Midsize,regular unleaded,AUTOMATIC,rear wheel drive,275.0,6.0,19,14,38015
4,7088: Ford Mustang (2017),0.746340,9174,Ford,Shelby GT500,2012,Coupe,Midsize,premium unleaded (required),MANUAL,rear wheel drive,550.0,8.0,23,15,48810
5,7088: Ford Mustang (2017),0.739261,6611,BMW,M4,2017,Coupe,Midsize,premium unleaded (required),MANUAL,rear wheel drive,425.0,6.0,26,17,66200
6,7088: Ford Mustang (2017),0.736363,439,BMW,4 Series,2016,Coupe,Midsize,premium unleaded (required),MANUAL,rear wheel drive,240.0,4.0,34,22,41850



Recommendations for Ford F-150
Using input car: 4560: Ford F-150 (2017)


,input_car,similarity,row_id,Make,Model,Year,Vehicle Style,Vehicle Size,Engine Fuel Type,Transmission Type,Driven_Wheels,Engine HP,Engine Cylinders,highway MPG,city mpg,MSRP
0,4560: Ford F-150 (2017),0.876982,10734,Ford,Transit Wagon,2017,Passenger Van,Large,flex-fuel (unleaded/E85),AUTOMATIC,rear wheel drive,275.0,6.0,19,14,35825
1,4560: Ford F-150 (2017),0.823347,3736,Ford,E-Series Van,2014,Cargo Van,Large,flex-fuel (unleaded/E85),AUTOMATIC,rear wheel drive,255.0,8.0,16,12,33660
2,4560: Ford F-150 (2017),0.751834,3754,Ford,E-Series Wagon,2014,Passenger Van,Midsize,flex-fuel (unleaded/E85),AUTOMATIC,rear wheel drive,255.0,8.0,15,11,33560
3,4560: Ford F-150 (2017),0.732641,9622,Chevrolet,Silverado 1500,2017,Regular Cab Pickup,Large,flex-fuel (unleaded/E85),AUTOMATIC,rear wheel drive,285.0,6.0,24,18,27585
4,4560: Ford F-150 (2017),0.730590,3138,Ford,Crown Victoria,2011,Sedan,Large,flex-fuel (unleaded/E85),AUTOMATIC,rear wheel drive,239.0,8.0,24,16,29905
5,4560: Ford F-150 (2017),0.724752,11036,Toyota,Tundra,2017,Regular Cab Pickup,Large,flex-fuel (unleaded/E85),AUTOMATIC,rear wheel drive,381.0,8.0,18,13,30400
6,4560: Ford F-150 (2017),0.717623,10301,Ford,Taurus,2017,Sedan,Large,flex-fuel (unleaded/E85),AUTOMATIC,front wheel drive,288.0,6.0,27,18,27345



Recommendations for Porsche 911
Using input car: 977: Porsche 911 (2017)


,input_car,similarity,row_id,Make,Model,Year,Vehicle Style,Vehicle Size,Engine Fuel Type,Transmission Type,Driven_Wheels,Engine HP,Engine Cylinders,highway MPG,city mpg,MSRP
0,977: Porsche 911 (2017),0.990622,2391,Porsche,Cayman,2016,Coupe,Compact,premium unleaded (required),MANUAL,rear wheel drive,340.0,6.0,26,19,75200
1,977: Porsche 911 (2017),0.947219,737,Porsche,718 Cayman,2017,Coupe,Compact,premium unleaded (required),MANUAL,rear wheel drive,350.0,4.0,26,20,66300
2,977: Porsche 911 (2017),0.916727,341,Nissan,370Z,2017,Coupe,Compact,premium unleaded (required),MANUAL,rear wheel drive,332.0,6.0,26,18,37970
3,977: Porsche 911 (2017),0.912375,4267,Lotus,Evora,2014,Coupe,Compact,premium unleaded (required),MANUAL,rear wheel drive,345.0,6.0,26,17,79980
4,977: Porsche 911 (2017),0.903392,2008,Porsche,Boxster,2016,Convertible,Compact,premium unleaded (required),MANUAL,rear wheel drive,330.0,6.0,26,19,74600
5,977: Porsche 911 (2017),0.903120,4257,Lotus,Evora 400,2017,Coupe,Compact,premium unleaded (required),MANUAL,rear wheel drive,400.0,6.0,39,21,91900
6,977: Porsche 911 (2017),0.885973,2384,Porsche,Cayman S,2006,Coupe,Compact,premium unleaded (required),MANUAL,rear wheel drive,295.0,6.0,26,18,58900



Recommendation systems often lack explicit labels for what is correct. Here I evaluate with proxy metrics:

- same vehicle style rate,
- same vehicle size rate,
- same fuel type rate,
- average normalized specification distance,
- comparison with a random recommender baseline.

A good content-based recommender should produce higher style/size/fuel matches and lower specification distance than random recommendations.

In [6]:
eval_numeric_cols = ['Engine HP', 'Engine Cylinders', 'highway MPG', 'city mpg', 'MSRP']
numeric_ranges = df[eval_numeric_cols].max() - df[eval_numeric_cols].min()
numeric_ranges = numeric_ranges.replace(0, 1)

def evaluate_rows(row_ids, top_n=10, random_seed=42):
    rng = random.Random(random_seed)
    rows = []

    for row_id in row_ids:
        target = df.loc[df['row_id'].eq(row_id)].iloc[0]
        recs = recommend_by_row_id(int(row_id), top_n=top_n)

        candidate_pool = df[df['row_id'].ne(row_id)].sample(n=top_n, random_state=rng.randint(1, 10_000))

        for method, candidates in [('content_based', recs), ('random_baseline', candidate_pool)]:
            spec_distance = (
                (candidates[eval_numeric_cols].reset_index(drop=True) - target[eval_numeric_cols].astype(float))
                .abs()
                .div(numeric_ranges, axis=1)
                .mean(axis=1)
                .mean()
            )

            rows.append({
                'input_car': target['car_label'],
                'method': method,
                'same_style_rate': (candidates['Vehicle Style'].values == target['Vehicle Style']).mean(),
                'same_size_rate': (candidates['Vehicle Size'].values == target['Vehicle Size']).mean(),
                'same_fuel_type_rate': (candidates['Engine Fuel Type'].values == target['Engine Fuel Type']).mean(),
                'avg_normalized_spec_distance': spec_distance,
                'avg_similarity': candidates['similarity'].mean() if 'similarity' in candidates.columns else np.nan,
                'unique_make_count': candidates['Make'].nunique(),
            })

    return pd.DataFrame(rows)

test_row_ids = [
    int(find_cars('Odyssey', make='Honda', limit=1).iloc[0]['row_id']),
    int(find_cars('Corolla', make='Toyota', limit=1).iloc[0]['row_id']),
    int(find_cars('Mustang', make='Ford', limit=1).iloc[0]['row_id']),
    int(find_cars('F-150', make='Ford', limit=1).iloc[0]['row_id']),
    int(find_cars('911', make='Porsche', limit=1).iloc[0]['row_id']),
]

evaluation = evaluate_rows(test_row_ids, top_n=10)
display(evaluation)

summary = evaluation.groupby('method').mean(numeric_only=True).sort_index()
display(summary)

,input_car,method,same_style_rate,same_size_rate,same_fuel_type_rate,avg_normalized_spec_distance,avg_similarity,unique_make_count
0,7305: Honda Odyssey (2016),content_based,0.8,0.4,1.0,0.009756,0.815684,8
1,7305: Honda Odyssey (2016),random_baseline,0.0,0.2,0.6,0.037362,NaN,10
2,2951: Toyota Corolla (2017),content_based,1.0,1.0,1.0,0.007112,0.980202,8
3,2951: Toyota Corolla (2017),random_baseline,0.0,0.5,0.4,0.077485,NaN,9
4,7088: Ford Mustang (2017),content_based,0.6,0.7,0.5,0.036972,0.752043,3
5,7088: Ford Mustang (2017),random_baseline,0.0,0.3,0.6,0.051510,NaN,8
6,4560: Ford F-150 (2017),content_based,0.4,0.8,0.8,0.027719,0.747231,4
7,4560: Ford F-150 (2017),random_baseline,0.0,0.3,0.1,0.038010,NaN,8
8,977: Porsche 911 (2017),content_based,0.9,1.0,0.9,0.018269,0.906836,5
9,977: Porsche 911 (2017),random_baseline,0.2,0.4,0.1,0.055406,NaN,8


,same_style_rate,same_size_rate,same_fuel_type_rate,avg_normalized_spec_distance,avg_similarity,unique_make_count
method,,,,,,
content_based,0.74,0.78,0.84,0.019965,0.840399,5.6
random_baseline,0.04,0.34,0.36,0.051954,NaN,8.6


The recommender is content-based because it does not use user ratings or purchase history. It compares cars by their own attributes. The evaluation compares it with random recommendations. If the content-based row has better style/size/fuel match rates and lower normalized spec distance, the recommender is behaving sensibly.